# Demonstrating Visualization of OPERA DIST-ANN Product Layers
---
This notebook provides an overview of the basic functionality and utility of the OPERA **DIST-ANN** product, an near-global annual pixel-wise summary of vegetation change. Here, several of the available **DIST-ANN** rasters are visualized for a wildfire-affected area in northern California.

**<font color='red'>Note: Please refer to [DIST product specification](https://d2pn8kiwq2w21t.cloudfront.net/documents/ProductSpec_DIST_HLS.pdf) for more information. </font>**

In [ ]:
### Library Imports
import earthaccess
import hvplot.xarray
import geoviews as gv
import holoviews as hv
import numpy as np
from bokeh.models import FixedTicker

hv.extension('bokeh')

import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../../')
from src.dist_utils import *

### DIST Product Suite Background
---
The land Disturbance product suite (**DIST**) maps vegetation disturbance from Harmonized Landsat-8 and Sentinel-2 A/B (HLS) scenes. Disturbance is detected when vegetation cover decreases or spectral variation is outside a historical norm within an HLS pixel. Two DIST products compose the DIST product suite: 1) the **DIST-ALERT** product, capturing vegetation disturbance at the cadence of HLS sampling (2-3 days); and 2) the **DIST-ANN** product, summarizing the confirmed changes of the DIST-ALERT products from previous calendar year. 

This notebook provides a step-by-step workflow visualizing **DIST-ANN** raster layers for the 2022 calendar year. An analogous notebook for the **DIST-ALERT** product may be accessed [here](https://github.com/OPERA-Cal-Val/OPERA_Applications/blob/main/DIST/Wildfire/Intro_To_DIST.ipynb).

### Metadata
---

HLS products provide surface reflectance (SR) data from the Operational Land Imager (OLI) aboard the Landsat-8 remote sensing satellite and the Multi-Spectral Instrument (MSI) aboard the Sentinel-2 A/B remote sensing satellite. HLS products are distributed over projected map coordinates aligned with the Military Grid Reference System (MGRS). Each tile covers 109.8 kilometers squared divided into 3660 rows and 3660 columns at 30 meter pixel spacing. Each tile overlaps neighbors by 4900 meters in each direction.

### Raster Layers
___

The **DIST-ANN** product is distributed as a set of 16 Cloud-Optimized GeoTIFF (COG) files to enable download of only particular layers of interest to a given user. All L3 DIST layers are stored in files following GeoTIFF format specifications. Details specific to the available raster layers and their properties are available in the [OPERA DIST Product Specifications Document](https://d2pn8kiwq2w21t.cloudfront.net/documents/ProductSpec_DIST_HLS.pdf).

## Authentication with NASA Earthdata credentials
A [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account is required to download the data used in this tutorial. You can create an account at the link provided. After establishing an account, the code in the next cell will verify authentication. If this is your first time running the notebook, you will be prompted to enter your Earthdata login credentials, which will be saved in ~/.netrc.

In [ ]:
auth = earthaccess.login(strategy="netrc")
s3_credentials = auth.get_s3_credentials(daac="PODAAC")

In [ ]:
# Specify filepath location of OPERA DIST_ANN tile and desired layers (Note: layer_names is not comprehensive of all available layers)
product = "https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/OPERA_L3_DIST-ANN-HLS_V1/OPERA_L3_DIST-ANN-HLS_T10TEM_2023_20240216T190732Z_30_v1/OPERA_L3_DIST-ANN-HLS_T10TEM_2023_20240216T190732Z_30_v1_"
layer_names = [
    'VEG-DIST-STATUS', 'VEG-HIST', 'VEG-IND-MAX', 'VEG-ANOM-MAX', 'VEG-DIST-CONF', 'VEG-DIST-DATE',
    'VEG-DIST-COUNT', 'VEG-DIST-DUR', 'VEG-LAST-DATE', 'GEN-DIST-STATUS', 'GEN-ANOM-MAX', 'GEN-DIST-CONF',
    'GEN-DIST-DATE', 'GEN-DIST-COUNT', 'GEN-DIST-DUR', 'GEN-LAST-DATE'
]
layer_paths = [f"{product}{layer}.tif" for layer in layer_names]
layers = earthaccess.open(layer_paths, provider='LPDAAC')


In [ ]:
# Create geocube of stacked bands
da, crs = stack_layers(layers)

# Create basemap
base = gv.tile_sources.EsriTerrain.opts(width=1000, height=1000, padding=0.1)

## **Band 1: Vegetation Disturbance Status (VEG-DIST-STATUS)**
***

**Data Type:** UInt8<br>
**Description:** Status of confirmed disturbance, current provisional disturbance, and no disturbance.<br>

In [ ]:
import numpy as np
from bokeh.models import FixedTicker

color_key = {
    3: "#dee043",
    6: "#e01b07",
    7: "#777777",
    8: "#dddddd",
    9: "#777777",
    10: "#dddddd"
}

labels_txt = {
    3: "Confirmed, <50% ongoing",
    6: "Confirmed, ≥50% ongoing",
    7: "Confirmed, <50% completed",
    8: "Confirmed, ≥50% completed",
    9: "Confirmed, prev yr <50%",
    10: "Confirmed, prev yr ≥50%",
}

levels = sorted(color_key)
code_to_index   = {c: i for i, c in enumerate(levels)}
index_to_color  = [color_key[c] for c in levels]
index_to_label  = {i: labels_txt[c] for i, c in enumerate(levels)}

veg_dist_status = da.z.where(da['z'] != 255).sel({'layer': 1})
veg_idx         = veg_dist_status.where(veg_dist_status != 0).copy()
veg_idx.data    = np.vectorize(code_to_index.get)(veg_idx.data)

N = len(index_to_color)
ticks  = list(range(N))
ticker = FixedTicker(ticks=ticks)
clim   = (-0.5, N - 0.5)

(veg_idx.hvplot.image(
        x='longitude',
        y='latitude',
        crs=crs,
        frame_width=500,
        frame_height=500,
        aspect='equal',
        cmap=index_to_color,
        clim=clim,
        alpha=0.8)
 .opts(title="VEG_DIST_STATUS",
       xlabel='Longitude', ylabel='Latitude',
       colorbar_opts={'ticker': ticker,
                      'major_label_overrides': index_to_label})
 * base)

In [ ]:
base = gv.tile_sources.EsriNatGeo.opts(width=1000, height=1000, padding=0.1)
veg_dist_status = da.z.where(da['z']!=255).sel({'layer':1})

color_key = {
    "3: Confirmed, <50% ongoing": "#dee043",
    "6: Confirmed, ≥50% ongoing": "#e01b07",
    "7: Confirmed, <50% completed": "#777777",
    "8: Confirmed, ≥50% completed": "#dddddd",
    "9: Confirmed, previous year <50%": "#777777",
    "10: Confirmed, previous year ≥50%": "#dddddd"
}

levels = 6
ticks = [3, 6, 7, 8, 9, 10]
ticker = FixedTicker(ticks=ticks)
labels = dict(zip(ticks, color_key))

veg_dist_status.where(veg_dist_status != 0).hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    clim=(2, 6),
    alpha=0.8
).opts(
    title="VEG_DIST_STATUS",
    xlabel='Longitude',
    ylabel='Latitude',
    color_levels=levels,
    cmap=tuple(color_key.values()),
    colorbar_opts={'ticker': ticker, 'major_label_overrides': labels}
) * base

**Layer Values:**<br> 
* **0:** No disturbance<br>
* **2:** Confirmed disturbance with vegetation cover change <50% (ongoing) <br>
* **4:** Confirmed disturbance with vegetation cover change ≥50% (ongoing) <br>
* **5:** Confirmed disturbance with vegetation cover change <50% (completed) <br>
* **6:** Confirmed disturbance with vegetation cover change ≥50% (completed)  <br>
* **255:** NoData <br> 

## **Band 4: Maximum Vegetation Anomaly Value (VEG_ANOM_MAX)**
***

**Data Type:** UInt8<br>
**Description:** Difference between historical vegetation cover and vegetation cover at the date of maximum decrease (vegetation loss of 0- 100%). This layer can be used to threshold vegetation disturbance per a given sensitivity (e.g. disturbance of ≥20% vegetation cover loss).<br>

In [ ]:
base = gv.tile_sources.EsriNatGeo.opts(width=1000, height=1000, padding=0.1)

# Select the 4th layer and mask out nodata
veg_anom_max = da.z.where(da['z'] != 255).sel({'layer': 4})
veg_anom_max = veg_anom_max.where(veg_anom_max != 0)

# Plot without rasterize
veg_anom_max.hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    cmap='hot',
    clabel='Vegetation Loss (%)',
    clim=(0, 100),
    alpha=0.8
).opts(
    title="VEG_ANOM_MAX",
    xlabel='Longitude',
    ylabel='Latitude'
).redim.nodata(value=255) * base

**Layer Values:**<br> 
* **0-100:** Maximum loss of percent vegetation<br>
* **255:** NoData <br>


## **Band 5: Vegetation Disturbance Confidence (VEG_DIST_CONF)**
***

**Data Type:** UInt16<br>
**Description:** Mean anomaly value since initial anomaly detection times the number of loss anomalies squared, until the anniversary date is reached, or a fixed number of consecutive non- anomalies are observed.<br>

In [ ]:
base = gv.tile_sources.EsriNatGeo.opts(width=1000, height=1000, padding=0.1)

veg_dist_confidence = da.z.where(da['z'] != 255).sel({'layer': 5})
veg_dist_confidence = veg_dist_confidence.where(veg_dist_confidence != 0)

veg_dist_confidence.hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    cmap='blues',
    clabel='Confidence Units',
    alpha=0.8,
    clim=(34, 32000)
).opts(
    title="VEG_DIST_CONFIDENCE",
    colorbar_opts={'ticker': FixedTicker(ticks=[0, 10000, 20000, 30000])},
    xlabel='Longitude',
    ylabel='Latitude'
) * base

**Layer Values:**<br> 
* **-1:** NoData <br>
* **0:** No disturbance <br>
* **>0:** Disturbance confidence <br>

## **Band 6: Date of Initial Vegetation Disturbance (VEG_DIST_DATE)**
***

**Data Type:** Int16<br>
**Description:** Day of first loss anomaly detection in the last year, denoted as the number of days since December 31st, 2020.<br>

In [ ]:
veg_dist_date = da.z.where(da['z'] != -1).sel({'layer': 6})
veg_dist_date = veg_dist_date.where(veg_dist_date != 0)

veg_dist_date.hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    cmap='inferno',
    clabel='Days since 1/1/22',
    alpha=0.8,
    clim=(0, 592)
).opts(
    title="VEG_DIST_DATE",
    xlabel='Longitude',
    ylabel='Latitude'
) * base

**Layer Values:**<br> 
* **-1:** NoData <br>
* **0:** No disturbance <br>
* **>0:** Day of first loss anomaly detection <br>

## **Band 7: Number of Vegetation Anomalies (VEG-DIST-COUNT)**
***

**Data Type:** UInt8<br>
**Description:** Total number of observations with anomalous low vegetation since initial anomaly detection (inclusive). Maximum of 254.<br>

In [ ]:
veg_dist_count = da.z.where(da['z'] != 255).sel({'layer': 7})
veg_dist_count = veg_dist_count.where(veg_dist_count != 0)

veg_dist_count.hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    cmap='cividis',
    clabel='Number of Anomalies Observed',
    alpha=0.8,
    clim=(0, 254)
).opts(
    title="VEG_DIST_COUNT",
    xlabel='Longitude',
    ylabel='Latitude',
    colorbar_opts={'ticker': FixedTicker(ticks=[0, 50, 100, 150, 200, 250])}
) * base

**Layer Values:**<br> 
* **0:** No disturbance anomalies <br>
* **1-254:** Count of disturbance anomalies <br>
* **255:** NoData <br>

## **Band 8: Vegetation Disturbance Duration (VEG-DIST-DUR)**
***

**Data Type:** UInt16<br>
**Description:** Number of days of ongoing loss anomalies since initial anomaly detection (inclusive). Maximum duration is one year.<br>

In [ ]:
veg_dist_dur = da.z.where(da['z'] != -1).sel({'layer': 8})
veg_dist_dur = veg_dist_dur.where(veg_dist_dur != 0)

veg_dist_dur.hvplot.image(
    x='longitude',
    y='latitude',
    crs=crs,
    dynamic=True,
    aspect='equal',
    frame_width=500,
    frame_height=500,
    cmap='magma_r',
    clabel='Days',
    alpha=0.8,
    clim=(0, 365)
).opts(
    title="VEG_DIST_DUR",
    xlabel='Longitude',
    ylabel='Latitude',
    colorbar_opts={'ticker': FixedTicker(ticks=[0, 50, 100, 150, 200, 250, 300, 350])}
) * base

**Layer Values:**<br> 
* **-1:** NoData <br>
* **0-366:** Number of days from first disturbance anomaly to the most recent disturbance anomaly detection <br>

## Conclusion
This notebook provides a basic workflows for loading and visualizing raster layers of the OPERA **DIST-ANN** product, a near-global, pixel-wise summary of vegetation loss for the 2022 calendar year. 